# v2 — 03: Pipeline & Cleaning
**Iteration 2** extends the v1 pipeline with:
- Draft capital features (pick, round, age at draft)
- Combine athleticism (40-yard dash, weight, vertical)
- Injury history (games missed, IR flag, career durability rate)
- College production (final season rec yards, TDs, dominator rate)
- **Rookies included** with NFL stat features set to 0

**Prerequisites:** Run `v1_01_data_gathering.ipynb` and `v2_01_data_gathering.ipynb` first.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from src.fetchers.nfl_fetcher import (
    fetch_seasonal_stats, fetch_rosters, fetch_snap_counts,
    fetch_draft_picks, fetch_combine_data, fetch_injuries,
)
from src.fetchers.sleeper_fetcher import fetch_players
from src.fetchers.college_fetcher import fetch_college_stats
from src.pipeline.cleaner import build_clean_dataset, read_from_db
from src.pipeline.features import build_model_dataset

pd.set_option('display.max_columns', 40)

## 1. Load All Data Sources (from cache)

In [ ]:
# v1 sources
seasonal     = fetch_seasonal_stats()
rosters      = fetch_rosters()
snap_counts  = fetch_snap_counts()
sleeper      = fetch_players()

# v2 sources
draft_picks  = fetch_draft_picks()
combine_data = fetch_combine_data()
injuries     = fetch_injuries()
college_stats = fetch_college_stats(draft_seasons=list(range(2016, 2025)))

print(f'Seasonal:     {seasonal.shape}')
print(f'Rosters:      {rosters.shape}')
print(f'Snap counts:  {snap_counts.shape}')
print(f'Draft picks:  {draft_picks.shape}')
print(f'Combine:      {combine_data.shape}')
print(f'Injuries:     {injuries.shape}')
print(f'College:      {college_stats.shape}')

## 2. Run v2 Cleaning Pipeline

In [ ]:
clean_df = build_clean_dataset(
    seasonal_df=seasonal,
    rosters_df=rosters,
    sleeper_df=sleeper,
    snap_df=snap_counts,
    draft_df=draft_picks,
    combine_df=combine_data,
    injury_df=injuries,
    college_df=college_stats,
)
print(f'\nClean dataset: {clean_df.shape}')
clean_df.head(3)

## 3. Inspect New Feature Coverage

In [ ]:
new_v2_cols = [
    'draft_pick', 'draft_round', 'age_at_draft', 'is_undrafted',
    'combine_forty', 'combine_weight', 'combine_vertical',
    'games_missed', 'ir_flag',
    'college_rec_yards', 'college_dominator_rate',
]
print('Coverage of new v2 features:')
for col in new_v2_cols:
    if col in clean_df.columns:
        pct = clean_df[col].notna().mean()
        print(f'  {col:35s}: {pct:.1%} non-null')
    else:
        print(f'  {col:35s}: NOT IN DATASET')

## 4. Feature Engineering (with Rookie Inclusion)

In [ ]:
model_df = build_model_dataset(clean_df)
print(f'\nModel-ready dataset: {model_df.shape}')
model_df.head(3)

## 5. Validate Rookie Inclusion

In [ ]:
if 'is_rookie' in model_df.columns:
    rookies = model_df[model_df['is_rookie'] == 1]
    vets = model_df[model_df['is_rookie'] == 0]
    print(f'Rookies in model dataset: {len(rookies)}')
    print(f'Veterans in model dataset: {len(vets)}')
    print(f'\nRookie ppr_pts_prev1 (should all be 0): {rookies["ppr_pts_prev1"].unique()}')
    print(f'\nSample rookies:')
    cols = ['full_name', 'position', 'season', 'draft_pick', 'age_at_draft',
            'college_rec_yards', 'ppr_pts_next']
    print(rookies[[c for c in cols if c in rookies.columns]].head(10).to_string(index=False))

## 6. Null Rates in Model Features

In [ ]:
null_rates = model_df.isna().mean().sort_values(ascending=False)
print('Feature null rates (top 20):')
print(null_rates[null_rates > 0].head(20))